# 04 - Imputation as a Supervised Problem

Missing data is where most analytics pipelines quietly go wrong. The usual
fix is a forward fill, chosen because it is one line, and almost never
measured.

**This dataset lets you measure it.** The analyst tree carries gaps from
injected communications outages; the truth tree has every value. So every
missing point has a known correct answer, in physical units, requiring no
annotation at all.

That is unusual. Most public data gives you either labels or realism, and
here you have both.

In [1]:

import pandas as pd
from northstar_analytics import find_dataset, open_dataset, score_imputation
from northstar_analytics.imputation import find_gaps, impute

DATASET = find_dataset("curriculum")
RUN_ID = "curriculum"

analyst = open_dataset(DATASET, RUN_ID, "analyst")
truth = open_dataset(DATASET, RUN_ID, "truth")

## 1. Find the gaps

Note what a gap is here: rows that are **present** with NULL fields, not
missing timestamps.

That distinction matters more than it sounds. TimescaleDB's `locf` and
`time_bucket_gapfill` fill buckets that *gapfill created* — and these buckets
already exist, so those functions cannot see them at all. Counting NULLs is
the only thing that works.

In [2]:

gaps = find_gaps(analyst)
print(f"{len(gaps)} gaps found\n")
print(gaps.head(10).to_string(index=False))

9 gaps found

           asset_id                     start                       end  minutes
NORTHSTA-BLK05-INV4 2023-06-24 11:58:00+00:00 2023-06-24 14:47:00+00:00      170
NORTHSTA-BLK07-INV3 2023-06-23 20:36:00+00:00 2023-06-23 22:49:00+00:00      134
NORTHSTA-BLK10-INV1 2023-06-23 03:38:00+00:00 2023-06-23 05:41:00+00:00      124
NORTHSTA-BLK03-INV1 2023-06-27 05:02:00+00:00 2023-06-27 06:41:00+00:00      100
NORTHSTA-BLK02-INV4 2023-06-23 23:50:00+00:00 2023-06-24 01:23:00+00:00       94
NORTHSTA-BLK08-INV3 2023-06-26 08:00:00+00:00 2023-06-26 09:32:00+00:00       93
NORTHSTA-BLK01-INV3 2023-06-26 21:07:00+00:00 2023-06-26 21:43:00+00:00       37
NORTHSTA-BLK05-INV2 2023-06-26 10:14:00+00:00 2023-06-26 10:45:00+00:00       32
NORTHSTA-BLK01-INV4 2023-06-23 06:08:00+00:00 2023-06-23 06:23:00+00:00       16


## 2. Look at one before modelling anything

Always plot the thing before you model it. The shape tells you which methods
have a chance.

In [3]:

worst = gaps.iloc[0]
window = analyst.execute(
    f"""
    SELECT time, asset_id, ac_power_kw, poa_global
    FROM inverter_telemetry
    WHERE asset_id = '{worst.asset_id}'
      AND time BETWEEN TIMESTAMP '{worst.start}' - INTERVAL '2 hours'
                   AND TIMESTAMP '{worst.end}' + INTERVAL '2 hours'
    ORDER BY time
    """
).df()

print(f"asset {worst.asset_id}, gap of {worst.minutes} minutes")
print(f"  rows in window        {len(window)}")
print(f"  non-null ac_power_kw  {window['ac_power_kw'].notna().sum()}")
print(f"  non-null poa_global   {window['poa_global'].notna().sum()}")

asset NORTHSTA-BLK05-INV4, gap of 170 minutes
  rows in window        410
  non-null ac_power_kw  410
  non-null poa_global   410


Every field is null together. A communications outage takes the whole
telemetry stream, not one channel — so you cannot impute power from that
inverter's own irradiance. The information has to come from somewhere else.

## 3. Where the information actually is

The other 39 inverters were reporting throughout. If they correlate with the
gapped one, they carry what is missing.

Check that before assuming it.

In [4]:

wide = (
    analyst.execute("SELECT time, asset_id, ac_power_kw FROM inverter_telemetry")
    .df()
    .pivot_table(index="time", columns="asset_id", values="ac_power_kw")
)

target = worst.asset_id
peers = wide.drop(columns=[target])
observed = wide[target].notna()

correlations = peers[observed].corrwith(wide.loc[observed, target])
print("correlation of the gapped inverter with its peers:")
print(f"  median {correlations.median():.4f}")
print(f"  min    {correlations.min():.4f}")
print(f"  max    {correlations.max():.4f}")

correlation of the gapped inverter with its peers:
  median 0.9999
  min    0.9823
  max    1.0000


Correlations near 1.0. That is the whole reason peer methods work here, and
it is a property of **this domain** rather than a general truth about
imputation — every asset sees nearly the same sun at nearly the same moment.

An interviewer asking "why did that work?" wants this answer, not the name of
the algorithm.

## 4. Score four methods against truth

* **forward fill** — carry the last value. What most pipelines do.
* **linear** — interpolate across the gap.
* **peer median** — the fleet median, rescaled by this asset's usual ratio.
* **peer regression** — least squares against the peers, fitted only on
  intervals where the target was reporting.

In [5]:

result = score_imputation(DATASET, RUN_ID)
scores = result.frame()
print(scores.to_string(index=False))

         method           column    n     mae    rmse     bias
peer_regression      ac_power_kw  800   3.191   5.833   -1.745
    peer_median      ac_power_kw  800   5.758  13.153   -1.471
         linear      ac_power_kw  800  53.541 102.490    9.400
   forward_fill      ac_power_kw  800 348.964 741.513 -127.298
    peer_median cell_temperature  800   0.306   0.418   -0.053
peer_regression cell_temperature  800   0.320   0.413   -0.062
         linear cell_temperature  800   0.728   1.277   -0.156
   forward_fill cell_temperature  800   4.012   7.251   -1.369
peer_regression      dc_power_kw  800   2.926   5.631   -0.130
    peer_median      dc_power_kw  800   6.673  14.031   -0.432
         linear      dc_power_kw  800  51.249 101.536   13.392
   forward_fill      dc_power_kw  800 375.922 757.895 -107.413
peer_regression       poa_global 1145   1.603   3.312   -0.666
    peer_median       poa_global 1145   2.357   5.610   -0.630
         linear       poa_global 1145  13.069  30.939  

## 5. Read the bias column, not just the error

MAE says how wrong. **Bias says which way**, and that is often the more
actionable number.

In [6]:

for column in scores["column"].unique():
    best = result.best(column)
    worst_method = max(
        (s for s in result.scores if s.column == column), key=lambda s: s.mae
    )
    print(
        f"{column:<18} best {best.method:<16} MAE {best.mae:>8.3f}   "
        f"{worst_method.mae / best.mae:>6.1f}x better than {worst_method.method}"
    )

print()
ff = [s for s in result.scores if s.method == "forward_fill"]
for score in ff:
    print(f"forward fill on {score.column:<18} bias {score.bias:>+9.3f}")

ac_power_kw        best peer_regression  MAE    3.191    109.4x better than forward_fill
cell_temperature   best peer_median      MAE    0.306     13.1x better than forward_fill
dc_power_kw        best peer_regression  MAE    2.926    128.5x better than forward_fill
poa_global         best peer_regression  MAE    1.603     59.1x better than forward_fill

forward fill on poa_global         bias   -26.395
forward fill on ac_power_kw        bias  -127.298
forward fill on dc_power_kw        bias  -107.413
forward fill on cell_temperature   bias    -1.369


Forward fill's bias is large and **negative** on the power columns. It
carries whatever the inverter was doing before it went quiet — and outages
that start near sunset carry the overnight standby draw straight through the
following daylight.

`sql/timeseries/02_gaps.sql` shows this concretely: −0.7 kW held across three
hours of full sun.

## 6. Does accuracy depend on gap length?

A method that handles five minutes well may fail over three hours. Averaging
across both hides it.

In [7]:

imputed, actual = impute(analyst, truth, "ac_power_kw")
errors = pd.DataFrame({name: (values - actual).abs() for name, values in imputed.items()})
errors["time"] = [index[0] for index in actual.index]
errors["asset"] = [index[1] for index in actual.index]

gap_lengths = {(row.asset_id): row.minutes for row in gaps.itertuples()}
errors["gap_minutes"] = errors["asset"].map(gap_lengths)
errors["bucket"] = pd.cut(
    errors["gap_minutes"],
    bins=[0, 30, 90, 180, 10_000],
    labels=["<30 min", "30-90", "90-180", ">180"],
)

print(
    errors.groupby("bucket", observed=True)[
        ["forward_fill", "linear", "peer_median", "peer_regression"]
    ]
    .mean()
    .round(2)
    .to_string()
)

         forward_fill  linear  peer_median  peer_regression
bucket                                                     
<30 min          0.05    0.05         0.05             0.03
30-90            5.02    3.53         2.83             2.84
90-180         389.96   59.56         6.17             3.29


Forward fill and linear degrade sharply with length — they extrapolate from
the edges, and the edges get further away. The peer methods barely care,
because they read the current interval rather than a past one.

**That is the argument for them in production**, and it is stronger than the
headline MAE.

## 7. Where to take this

1. **Beat peer regression.** Gradient boosting over engineered features —
   peer median, solar zenith, time of day, the target's historical ratio to
   its block. Score it the same way. It may not win, and finding that out is
   a legitimate result.
2. **Impute `operating_state`.** Categorical, so a classifier, and the metric
   changes to accuracy and confusion rather than MAE.
3. **Try it with fewer peers.** How many reporting inverters do you need?
   That question decides whether the method survives a site-wide outage.
4. **Then move to fault classification.** `scenario_instances` carries the
   class label, and `northstar-sim score` gives you a baseline at 39.2%
   recall and 81.7% precision. Same labelled-truth trick, harder problem.

In [8]:

analyst.close()
truth.close()